# 🎬 Binary Text Classification — IMDB Dataset (CNN)

**Objectif :** Classifier les critiques de films comme **positives (1)** ou **négatives (0)** à l'aide d'un réseau de neurones feedforward.

**Dataset :** IMDB Movie Reviews — 50 000 critiques, encodées comme séquences d'entiers (10 000 mots les plus fréquents).

## 1. Imports & Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# Reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")

## 2. Chargement du Dataset IMDB

In [ ]:
NUM_WORDS = 10_000  # Garder les 10 000 tokens les plus fréquents

(train_data, train_labels), (test_data, test_labels) = keras.datasets.imdb.load_data(
    num_words=NUM_WORDS
)

print(f"Données d'entraînement : {len(train_data)} critiques")
print(f"Données de test        : {len(test_data)} critiques")
print(f"\nExemple de critique encodée (première) :")
print(train_data[0][:20], "...")
print(f"Label correspondant : {train_labels[0]} ({'Positif' if train_labels[0] == 1 else 'Négatif'})")
print(f"\nDistribution des labels (train) :")
print(f"  Positifs : {sum(train_labels == 1)} | Négatifs : {sum(train_labels == 0)}")

## 3. Prétraitement des Données

### 3.1 Vectorisation One-Hot

On transforme chaque critique (liste d'entiers) en un vecteur binaire de taille 10 000 :
- **0** → le mot n'est pas présent dans la critique
- **1** → le mot est présent

> *Exemple : la séquence `[3, 5]` devient un vecteur de 10 000 dimensions, tout à 0 sauf aux indices 3 et 5 qui valent 1.*

In [ ]:
def vectorize_sequences(sequences, dimension=10_000):
    """Convertit des séquences d'entiers en matrices binaires (one-hot)."""
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1  # Active les indices correspondant aux mots
    return results

# Vectorisation
X_train_full = vectorize_sequences(train_data)
X_test       = vectorize_sequences(test_data)

# Labels en float32
y_train_full = train_labels.astype("float32")
y_test       = test_labels.astype("float32")

print(f"Forme X_train_full : {X_train_full.shape}")
print(f"Forme X_test       : {X_test.shape}")
print(f"Exemple vecteur (premiers 20 éléments) : {X_train_full[0][:20]}")

### 3.2 Split Train / Validation

In [ ]:
VAL_SIZE = 10_000  # 10 000 exemples pour la validation

X_val   = X_train_full[:VAL_SIZE]
X_train = X_train_full[VAL_SIZE:]

y_val   = y_train_full[:VAL_SIZE]
y_train = y_train_full[VAL_SIZE:]

print(f"Taille entraînement : {len(X_train):,} exemples")
print(f"Taille validation   : {len(X_val):,} exemples")
print(f"Taille test         : {len(X_test):,} exemples")

## 4. Construction du Modèle

Architecture : **2 couches cachées Dense + ReLU** → **couche de sortie Sigmoid**

| Couche | Unités | Activation | Rôle |
|--------|--------|------------|------|
| Dense  | 16     | ReLU       | Extraction de features |
| Dense  | 16     | ReLU       | Extraction de features |
| Dense  | 1      | Sigmoid    | Probabilité de classe positive |

In [ ]:
def build_model():
    model = keras.Sequential([
        keras.layers.Dense(16, activation="relu", input_shape=(NUM_WORDS,)),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1,  activation="sigmoid")  # Sortie : P(positive)
    ])
    
    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",  # Perte pour classification binaire
        metrics=["accuracy"]
    )
    return model

model = build_model()
model.summary()

## 5. Entraînement Initial (20 epochs)

On entraîne le modèle sur 20 epochs pour **observer l'overfitting**.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=512,
    validation_data=(X_val, y_val),
    verbose=1
)

print("\n✅ Entraînement terminé !")

## 6. Visualisation des Métriques d'Entraînement

In [ ]:
def plot_training_history(history, title_suffix=""):
    hist = history.history
    epochs = range(1, len(hist["loss"]) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Métriques d'entraînement {title_suffix}", fontsize=14, fontweight="bold")
    
    # --- Perte ---
    ax1.plot(epochs, hist["loss"],     "bo-", label="Perte entraînement", linewidth=2)
    ax1.plot(epochs, hist["val_loss"], "ro-", label="Perte validation",   linewidth=2)
    ax1.set_title("Perte (Loss)")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Binary Crossentropy")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Marquer le minimum de val_loss
    best_epoch = np.argmin(hist["val_loss"]) + 1
    ax1.axvline(x=best_epoch, color="green", linestyle="--", alpha=0.7,
                label=f"Meilleure epoch : {best_epoch}")
    ax1.legend()
    
    # --- Précision ---
    ax2.plot(epochs, hist["accuracy"],     "bo-", label="Précision entraînement", linewidth=2)
    ax2.plot(epochs, hist["val_accuracy"], "ro-", label="Précision validation",   linewidth=2)
    ax2.set_title("Précision (Accuracy)")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"/home/claude/training_metrics{title_suffix.replace(' ', '_')}.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    
    return best_epoch

best_epoch = plot_training_history(history, "— 20 epochs")
print(f"\n📌 Epoch optimale (val_loss minimale) : {best_epoch}")

## 7. Analyse de l'Overfitting

> **Observation attendue :**  
> - La perte d'entraînement continue de **diminuer** après ~4-5 epochs.  
> - La perte de validation commence à **remonter** → signe d'overfitting.  
> - L'epoch optimale correspond au **minimum de la perte de validation**.

In [ ]:
hist = history.history

print("📊 Résumé par epoch (loss entraînement vs validation)\n")
print(f"{'Epoch':>6} | {'Loss Train':>11} | {'Loss Val':>9} | {'Acc Train':>10} | {'Acc Val':>8}")
print("-" * 58)

for i in range(len(hist["loss"])):
    marker = " ← OPTIMAL" if i + 1 == best_epoch else ""
    print(f"{i+1:>6} | {hist['loss'][i]:>11.4f} | {hist['val_loss'][i]:>9.4f} | "
          f"{hist['accuracy'][i]:>10.4f} | {hist['val_accuracy'][i]:>8.4f}{marker}")

## 8. Réentraînement avec le Nombre d'Epochs Optimal

On réinitialise et réentraîne avec **`best_epoch`** epochs pour éviter l'overfitting.

In [ ]:
print(f"🔄 Réentraînement sur {best_epoch} epochs (epoch optimale)...\n")

# Nouveau modèle (réinitialisé)
final_model = build_model()

# Entraînement sur TOUTES les données train + val (pas de split cette fois)
final_history = final_model.fit(
    X_train_full, y_train_full,   # Train complet
    epochs=best_epoch,
    batch_size=512,
    verbose=1
)

print("\n✅ Modèle final entraîné !")

## 9. Évaluation Finale sur le Test Set

In [ ]:
test_loss, test_acc = final_model.evaluate(X_test, y_test, verbose=0)

print("=" * 45)
print("        RÉSULTATS FINAUX SUR LE TEST SET")
print("=" * 45)
print(f"  Loss (Binary Crossentropy) : {test_loss:.4f}")
print(f"  Accuracy                   : {test_acc:.4f} ({test_acc*100:.2f}%)")
print("=" * 45)
print(f"\n📌 Epochs utilisées : {best_epoch} (optimales)")

## 10. Prédictions sur Quelques Exemples

In [ ]:
# Quelques prédictions sur le jeu de test
sample_indices = [0, 1, 2, 3, 4]
predictions = final_model.predict(X_test[sample_indices], verbose=0)

print("Exemples de prédictions :\n")
print(f"{'Index':>6} | {'Vrai label':>11} | {'Proba pos.':>11} | {'Prédiction':>11} | {'Correct ?':>9}")
print("-" * 60)

for idx, pred in zip(sample_indices, predictions):
    true_label  = int(y_test[idx])
    proba       = float(pred[0])
    pred_label  = 1 if proba >= 0.5 else 0
    correct     = "✅" if pred_label == true_label else "❌"
    label_str   = "Positif" if true_label == 1 else "Négatif"
    pred_str    = "Positif" if pred_label == 1 else "Négatif"
    print(f"{idx:>6} | {label_str:>11} | {proba:>11.4f} | {pred_str:>11} | {correct:>9}")

## 11. Conclusion

### ✅ Ce que nous avons accompli

| Étape | Description |
|-------|-------------|
| **Prétraitement** | Vectorisation one-hot des séquences → matrices binaires (10 000 dims) |
| **Architecture** | Réseau feedforward : Dense(16, ReLU) → Dense(16, ReLU) → Dense(1, Sigmoid) |
| **Optimisation** | RMSprop + Binary Crossentropy |
| **Overfitting** | Détecté via courbes val_loss (remontée après epoch optimale) |
| **Solution** | Réentraînement avec nombre d'epochs optimal |

### 📈 Résultats attendus
- **Accuracy test** : ~88-89% (réseau feedforward simple sur représentation bag-of-words)  
- **Amélioration possible** : LSTM, GRU, ou Transformers pour capturer le contexte séquentiel

### 💡 Points clés retenus
1. Le **one-hot encoding** perd l'information d'ordre mais suffit pour une classification simple.
2. La **binary crossentropy** est la perte naturelle pour une sortie Sigmoid binaire.
3. L'**overfitting** se détecte quand `val_loss` remonte alors que `train_loss` continue de baisser.
4. Entraîner sur l'**epoch optimale** améliore la généralisation sur données inédites.